# Homework: Vector Search
## Q1. Embedding a query

In [2]:
from embedder import Embedder

embedder = Embedder()

In [3]:
query = "How does approximate nearest neighbor search work?"
query_vector = embedder.encode(query)

In [4]:
query_vector[0]

np.float64(-0.02058203437252893)

## Q2. Cosine similarity

In [5]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
  repo_owner="DataTalksClub",
  repo_name="llm-zoomcamp",
	commit_id="8c1834d",
	allowed_extensions={"md"},
	filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [6]:
content = ""

for document in documents:
  if document['filename'] == "02-vector-search/lessons/07-sqlitesearch-vector.md":
    content = document['content']
    break

In [7]:
document_vector = embedder.encode(content)

In [8]:
cosine_similarity = query_vector.dot(document_vector)
print(round(cosine_similarity, 2))

0.36


## Q3. Chunking and search by hand

In [9]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [10]:
import numpy as np
X = np.array(embedder.encode_batch([chunk['content'] for chunk in chunks]))

In [11]:
scores = X.dot(query_vector)

In [12]:
idx = np.argmax(scores)
chunks[idx]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

## Q4. Vector search with minsearch

In [13]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [14]:
query = "What metric do we use to evaluate a search engine?"
query_vector = embedder.encode(query)

results = vindex.search(query_vector, num_results=5)

In [15]:
results[0]['filename']

'04-evaluation/lessons/05-search-metrics.md'

## Q5. Text search vs vector search

In [16]:
from minsearch import Index

index = Index(
	text_fields=['content'],
	keyword_fields=['filename']
)

index.fit(chunks)

In [17]:
question = "How do I store vectors in PostgreSQL?"

In [18]:
text_search_results = index.search(question, num_results=5)

In [19]:
query_vector = embedder.encode(question)
vector_search_results = vindex.search(query_vector, num_results=5)

In [20]:
[result['filename'] for result in text_search_results]

['02-vector-search/lessons/02-embeddings.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md']

In [21]:
[result['filename'] for result in vector_search_results]

['02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md']

## Q6. Hybrid search

In [22]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [23]:
query = "How do I give the model access to tools?"
query_vector = embedder.encode(query)

In [24]:
vector_results = vindex.search(query_vector, num_results=5)
text_results = index.search(query, num_results=5)

In [25]:
results = rrf([vector_results, text_results])
results[0]

{'start': 4000,
 'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function ca